In [ ]:
# Cell 1 — Imports
import mlflow
import mlflow.pytorch
import mlflow.transformers
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    pipeline
)
from torch.utils.data import (
    DataLoader, TensorDataset
)
from torch.optim import AdamW
from transformers import (
    get_linear_schedule_with_warmup
)
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report,
    confusion_matrix
)
from dotenv import load_dotenv
import boto3

load_dotenv()

# Device
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu')
print(f"✅ Device: {device}")

In [ ]:
# Cell 2 — Configure MLflow
MLFLOW_URI        = os.getenv(
    'MLFLOW_TRACKING_URI',
    'http://localhost:5001')
EXPERIMENT_NAME   = os.getenv(
    'MLFLOW_EXPERIMENT_NAME',
    'document-ai-classification')

# Point MLflow to our tracking server
mlflow.set_tracking_uri(MLFLOW_URI)

# Create or get experiment
experiment = mlflow.set_experiment(
    EXPERIMENT_NAME)

print(f"✅ MLflow configured")
print(f"   Tracking URI:    {MLFLOW_URI}")
print(f"   Experiment:      {EXPERIMENT_NAME}")
print(f"   Experiment ID:   {experiment.experiment_id}")
print(f"\n   Open UI at: {MLFLOW_URI}")

In [ ]:
# Cell 3 — Load everything needed
print("Loading data and models...")

# Load tensors
train_data = torch.load(
    '../data/processed/train_tensors.pt',
    map_location='cpu')
val_data   = torch.load(
    '../data/processed/val_tensors.pt',
    map_location='cpu')
test_data  = torch.load(
    '../data/processed/test_tensors.pt',
    map_location='cpu')

with open(
    '../data/processed/prep_config.json'
) as f:
    config = json.load(f)

BATCH_SIZE  = config['batch_size']
NUM_CLASSES = config['num_classes']
MAX_LENGTH  = config['max_length']
MODEL_NAME  = config['model_name']
LABEL_MAP   = {int(k): v for k, v
               in config['label_map'].items()}
LABEL_NAMES = list(LABEL_MAP.values())

def make_dataloader(data_dict,
                    batch_size,
                    shuffle=False):
    dataset = TensorDataset(
        data_dict['input_ids'],
        data_dict['attention_mask'],
        data_dict['labels']
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0
    )

train_loader = make_dataloader(
    train_data, BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(
    val_data, BATCH_SIZE)
test_loader  = make_dataloader(
    test_data, BATCH_SIZE)

# Load previous results for comparison
with open('../logs/bert_results.json') as f:
    previous_results = json.load(f)

print(f"✅ Data loaded")
print(f"   Previous best accuracy: "
      f"{previous_results['test_accuracy']}")
print(f"   Previous best F1:       "
      f"{previous_results['test_f1_macro']}")

In [ ]:
# Cell 4 — Training + evaluation functions
def evaluate_model(model, loader,
                   criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            ids, mask, labels = [
                b.to(device) for b in batch]
            outputs = model(
                input_ids=ids,
                attention_mask=mask)
            loss = criterion(
                outputs.logits, labels)
            total_loss += loss.item()
            preds = outputs.logits.argmax(
                dim=1)
            all_preds.extend(
                preds.cpu().numpy())
            all_labels.extend(
                labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    return {
        'loss':     total_loss / len(loader),
        'accuracy': accuracy_score(
            all_labels, all_preds),
        'f1_macro': f1_score(
            all_labels, all_preds,
            average='macro'),
        'preds':    all_preds,
        'labels':   all_labels
    }


def train_with_mlflow(
        run_name,
        learning_rate,
        num_epochs,
        batch_size,
        warmup_ratio=0.1,
        weight_decay=0.01):
    """
    Complete training run with full MLflow logging.
    Every parameter, metric, and artifact tracked.
    """

    with mlflow.start_run(run_name=run_name) as run:

        run_id = run.info.run_id
        print(f"\n{'='*55}")
        print(f"  MLflow Run: {run_name}")
        print(f"  Run ID:     {run_id}")
        print(f"{'='*55}")

        # ── LOG PARAMETERS ──────────────────────
        # These are the knobs we turned
        mlflow.log_params({
            'model_name':     MODEL_NAME,
            'learning_rate':  learning_rate,
            'num_epochs':     num_epochs,
            'batch_size':     batch_size,
            'warmup_ratio':   warmup_ratio,
            'weight_decay':   weight_decay,
            'max_length':     MAX_LENGTH,
            'num_classes':    NUM_CLASSES,
            'optimizer':      'AdamW',
            'scheduler':      'linear_warmup',
            'device':         str(device),
            'dataset':        'AG News',
            'train_samples':  len(
                train_data['labels']),
            'val_samples':    len(
                val_data['labels']),
        })

        print("✅ Parameters logged to MLflow")

        # ── SETUP MODEL ──────────────────────────
        model = DistilBertForSequenceClassification\
            .from_pretrained(
                MODEL_NAME,
                num_labels=NUM_CLASSES
            ).to(device)

        criterion  = nn.CrossEntropyLoss()
        optimizer  = AdamW(
            model.parameters(),
            lr=learning_rate,
            eps=1e-8,
            weight_decay=weight_decay
        )

        total_steps  = len(train_loader) * \
                       num_epochs
        warmup_steps = int(
            total_steps * warmup_ratio)
        scheduler    = \
            get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )

        # ── TRAINING LOOP ────────────────────────
        best_val_f1 = 0
        best_epoch  = 0

        for epoch in range(1, num_epochs + 1):
            print(f"\n  Epoch {epoch}/{num_epochs}")

            # Train
            model.train()
            train_loss    = 0
            train_correct = 0
            train_total   = 0

            for batch in train_loader:
                ids, mask, labels = [
                    b.to(device) for b in batch]

                optimizer.zero_grad()
                outputs = model(
                    input_ids=ids,
                    attention_mask=mask)
                loss = criterion(
                    outputs.logits, labels)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0)

                optimizer.step()
                scheduler.step()

                train_loss    += loss.item()
                preds          = outputs.logits\
                    .argmax(dim=1)
                train_correct += (
                    preds == labels).sum().item()
                train_total   += labels.size(0)

            train_loss_avg = train_loss / \
                             len(train_loader)
            train_acc      = train_correct / \
                             train_total

            # Validate
            val_metrics = evaluate_model(
                model, val_loader,
                criterion, device)

            current_lr = scheduler\
                .get_last_lr()[0]

            print(f"    Train Loss: "
                  f"{train_loss_avg:.4f} | "
                  f"Train Acc: {train_acc:.4f}")
            print(f"    Val Loss:   "
                  f"{val_metrics['loss']:.4f} | "
                  f"Val Acc: "
                  f"{val_metrics['accuracy']:.4f}"
                  f" | Val F1: "
                  f"{val_metrics['f1_macro']:.4f}")

            # ── LOG METRICS PER EPOCH ────────────
            # step=epoch makes charts show
            # progression over time in MLflow UI
            mlflow.log_metrics({
                'train_loss':  train_loss_avg,
                'train_acc':   train_acc,
                'val_loss':    val_metrics['loss'],
                'val_accuracy':val_metrics[
                    'accuracy'],
                'val_f1_macro':val_metrics[
                    'f1_macro'],
                'learning_rate': current_lr,
            }, step=epoch)

            # Track best epoch
            if val_metrics['f1_macro'] > \
               best_val_f1:
                best_val_f1 = val_metrics[
                    'f1_macro']
                best_epoch  = epoch
                model.save_pretrained(
                    '../models/mlflow_best')

        # ── FINAL TEST EVALUATION ────────────────
        print("\n  Running final test evaluation...")

        best_model = \
            DistilBertForSequenceClassification\
            .from_pretrained(
                '../models/mlflow_best'
            ).to(device)

        test_metrics = evaluate_model(
            best_model, test_loader,
            criterion, device)

        print(f"\n  Test Accuracy: "
              f"{test_metrics['accuracy']:.4f}")
        print(f"  Test F1 Macro: "
              f"{test_metrics['f1_macro']:.4f}")

        # ── LOG FINAL TEST METRICS ───────────────
        mlflow.log_metrics({
            'test_accuracy':  test_metrics[
                'accuracy'],
            'test_f1_macro':  test_metrics[
                'f1_macro'],
            'best_val_f1':    best_val_f1,
            'best_epoch':     best_epoch,
        })

        # ── LOG CONFUSION MATRIX ─────────────────
        fig, ax = plt.subplots(figsize=(8, 6))
        cm = confusion_matrix(
            test_metrics['labels'],
            test_metrics['preds'])
        cm_pct = cm.astype('float') / \
                 cm.sum(axis=1)[:,np.newaxis] * 100

        sns.heatmap(
            cm_pct, annot=True, fmt='.1f',
            cmap='Blues', ax=ax,
            xticklabels=LABEL_NAMES,
            yticklabels=LABEL_NAMES)
        ax.set_title(
            f'{run_name}\n'
            f'Test Acc: '
            f'{test_metrics["accuracy"]:.4f} | '
            f'F1: '
            f'{test_metrics["f1_macro"]:.4f}')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
        plt.tight_layout()

        # Save and log as MLflow artifact
        cm_path = f'../logs/cm_{run_name}.png'
        plt.savefig(cm_path, dpi=150)
        plt.close()

        mlflow.log_artifact(cm_path,
                            'confusion_matrices')
        print("✅ Confusion matrix logged")

        # ── LOG CLASSIFICATION REPORT ────────────
        report = classification_report(
            test_metrics['labels'],
            test_metrics['preds'],
            target_names=LABEL_NAMES,
            output_dict=True)

        # Log per-class metrics
        for label_name in LABEL_NAMES:
            if label_name in report:
                mlflow.log_metrics({
                    f'{label_name}_precision':
                        report[label_name][
                            'precision'],
                    f'{label_name}_recall':
                        report[label_name][
                            'recall'],
                    f'{label_name}_f1':
                        report[label_name][
                            'f1-score'],
                })

        print("✅ Per-class metrics logged")

        # ── LOG MODEL ────────────────────────────
        # Save model as MLflow artifact
        mlflow.pytorch.log_model(
            best_model,
            artifact_path="distilbert_model",
            registered_model_name=
                f"document-classifier-{run_name}"
        )

        print("✅ Model logged to MLflow")
        print(f"\n  Run complete!")
        print(f"  Run ID: {run_id}")
        print(f"  View at: {MLFLOW_URI}")

        return {
            'run_id':         run_id,
            'run_name':       run_name,
            'test_accuracy':  test_metrics[
                'accuracy'],
            'test_f1_macro':  test_metrics[
                'f1_macro'],
            'best_val_f1':    best_val_f1,
            'best_epoch':     best_epoch,
            'params': {
                'learning_rate': learning_rate,
                'num_epochs':    num_epochs,
                'batch_size':    batch_size,
            }
        }

In [ ]:
# Cell 5 — Run experiments with different configs
print("Running MLflow experiments...")
print("Comparing 3 different configurations\n")

all_run_results = []

# ── Experiment 1: Baseline ───────────────────────
print("EXPERIMENT 1: Baseline configuration")
result1 = train_with_mlflow(
    run_name      = "baseline_lr2e5_ep3",
    learning_rate = 2e-5,
    num_epochs    = 3,
    batch_size    = 16,
    warmup_ratio  = 0.1,
    weight_decay  = 0.01
)
all_run_results.append(result1)

In [ ]:
# ── Experiment 2: Higher Learning Rate ──────────
print("\nEXPERIMENT 2: Higher learning rate")
result2 = train_with_mlflow(
    run_name      = "higher_lr_3e5_ep3",
    learning_rate = 3e-5,
    num_epochs    = 3,
    batch_size    = 16,
    warmup_ratio  = 0.1,
    weight_decay  = 0.01
)
all_run_results.append(result2)

In [ ]:
# ── Experiment 3: More Warmup ────────────────────
print("\nEXPERIMENT 3: More warmup steps")
result3 = train_with_mlflow(
    run_name      = "more_warmup_lr2e5_ep3",
    learning_rate = 2e-5,
    num_epochs    = 3,
    batch_size    = 16,
    warmup_ratio  = 0.2,
    weight_decay  = 0.01
)
all_run_results.append(result3)

In [ ]:
# Cell 6 — Compare experiments
print("\n" + "="*65)
print("  EXPERIMENT COMPARISON")
print("="*65)
print(f"{'Run Name':<30} {'Test Acc':>10} "
      f"{'Test F1':>10} {'Best Epoch':>12}")
print("-"*65)

best_result = None
best_f1     = 0

for result in all_run_results:
    print(f"{result['run_name']:<30} "
          f"{result['test_accuracy']:>10.4f} "
          f"{result['test_f1_macro']:>10.4f} "
          f"{result['best_epoch']:>12}")

    if result['test_f1_macro'] > best_f1:
        best_f1     = result['test_f1_macro']
        best_result = result

print("="*65)
print(f"\n🏆 Best Run: {best_result['run_name']}")
print(f"   Test F1:  {best_result['test_f1_macro']:.4f}")
print(f"   Run ID:   {best_result['run_id']}")
print(f"\nView all runs at: "
      f"http://localhost:5001")

In [ ]:
# Cell 7 — Use MLflow client to query runs
client = MlflowClient(
    tracking_uri=MLFLOW_URI)

experiment = client.get_experiment_by_name(
    EXPERIMENT_NAME)
exp_id = experiment.experiment_id

# Get all runs sorted by F1
runs = client.search_runs(
    experiment_ids=[exp_id],
    order_by=["metrics.test_f1_macro DESC"]
)

print(f"=== ALL MLFLOW RUNS ===")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Total runs: {len(runs)}\n")

for i, run in enumerate(runs):
    params  = run.data.params
    metrics = run.data.metrics

    print(f"Rank {i+1}: {run.data.tags.get('mlflow.runName', run.info.run_id[:8])}")
    print(f"  Run ID:        {run.info.run_id}")
    print(f"  Learning Rate: {params.get('learning_rate', 'N/A')}")
    print(f"  Epochs:        {params.get('num_epochs', 'N/A')}")
    print(f"  Test Accuracy: {metrics.get('test_accuracy', 0):.4f}")
    print(f"  Test F1:       {metrics.get('test_f1_macro', 0):.4f}")
    print(f"  Best Val F1:   {metrics.get('best_val_f1', 0):.4f}")
    print()

In [ ]:
# Cell 8 — Register best model in MLflow registry
print("Registering champion model...")

best_run_id   = best_result['run_id']
model_name    = "document-ai-champion"

# Register the model
model_uri = f"runs:/{best_run_id}/distilbert_model"

registered = mlflow.register_model(
    model_uri   = model_uri,
    name        = model_name
)

print(f"✅ Model registered")
print(f"   Name:    {model_name}")
print(f"   Version: {registered.version}")
print(f"   Run ID:  {best_run_id}")

# Transition to Production stage
client.transition_model_version_stage(
    name    = model_name,
    version = registered.version,
    stage   = "Production"
)

print(f"\n✅ Model promoted to Production stage")
print(f"   View registry at: "
      f"http://localhost:5001/#/models")